# Rewire the Room — Motor de Matching (notebook)

Roda o pipeline completo (normalize → zonas via CP-SAT → pods → rationale/starter)
em cima de qualquer planilha com a **mesma estrutura de colunas** da planilha de teste:

`ID, Função/Área, Setor, Interesses (2 a 3), Estágio da Org., Desafio Atual (Aberto), Opt-in Networking?, Interesse em Palestras (Múltipla Escolha), T1, T2, T3`

**Requisito de pasta:** este notebook precisa estar na MESMA pasta dos arquivos
`normalize.py`, `zones.py`, `scoring.py`, `optimizer.py`, `pods.py`, `rationale.py`,
`engine.py` — eles são importados como módulos, não reescritos aqui.


## 1. Setup — instalar dependências

In [ ]:
# Rode uma vez. No VSCode, selecione o kernel do seu venv/conda antes de rodar.
%pip install ortools pandas -q


## 2. Parâmetros — troque aqui para rodar com outra base

`CSV_PATH` pode apontar para qualquer planilha com a mesma estrutura de colunas.


In [ ]:
from pathlib import Path

CSV_PATH = "Hackathon_-_Tabela_para_testes.csv"   # <-- troque aqui para outra base
OUT_DIR = "out"

Path(OUT_DIR).mkdir(exist_ok=True)
assert Path(CSV_PATH).exists(), f"Arquivo nao encontrado: {CSV_PATH}"


## 3. Imports dos módulos do motor

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))  # garante que os .py da pasta sejam importaveis

import importlib
import normalize, zones, scoring, optimizer, pods, rationale, engine
for m in (normalize, zones, scoring, optimizer, pods, rationale, engine):
    importlib.reload(m)  # util se voce editar os .py e quiser re-rodar sem reiniciar o kernel

from engine import (
    run_pipeline, build_assignments_table, write_assignments_csv,
    build_explainability_report, build_pods_index,
)


## 4. Rodar o pipeline completo

In [ ]:
participants, by_zone, pods_by_zone, meta = run_pipeline(CSV_PATH)

print("Status do solver:", meta["solver_status"])
print("Modo do algoritmo:", meta["algorithm_mode"])
print("Runtime do solver (ms):", meta["solver_runtime_ms"])
print("Total:", meta["n_total"], "| opt-in:", meta["n_opt_in"], "| opt-out:", meta["n_opt_out"])
print()
print("Tamanho das zonas:", {z: len(by_zone[z]) for z in by_zone})
print("Pods por zona:", {z: len(pods_by_zone[z]) for z in pods_by_zone})


## 5. Tabela de assignments (participante → zona/pod/scores)

In [ ]:
import pandas as pd

rows = build_assignments_table(pods_by_zone)
write_assignments_csv(rows, f"{OUT_DIR}/assignments.csv")

df = pd.DataFrame(rows)
df.head(20)


## 6. Índice de pods (rationale + starter) — pronto para consulta

In [ ]:
import json

pods_index = build_pods_index(pods_by_zone)
with open(f"{OUT_DIR}/pods_index.json", "w", encoding="utf-8") as f:
    json.dump(pods_index, f, ensure_ascii=False, indent=2)

pd.DataFrame([
    {
        "pod_id": pid,
        "zona": p["zone"],
        "tamanho": p["size"],
        "total_score": round(p["scores"]["total"], 2) if p["scores"] else None,
        "starter": p["starter"],
    }
    for pid, p in pods_index.items()
]).sort_values(["zona", "pod_id"])


## 7. Consultar um pod específico

Troque `POD_ID` abaixo (veja a tabela acima para os IDs disponíveis, ex.: `A1`, `B3`, `OPEN`).

In [ ]:
POD_ID = "B2"

pod = pods_index[POD_ID]
print(f"POD {POD_ID}  —  Zona {pod['zone']}: {pod['zone_title']}")
print(f"Tensao: {pod['zone_tension']}")
print(f"Tamanho: {pod['size']}")
if pod["scores"]:
    s = pod["scores"]
    print(f"ConversationValue medio: {s['total']:.2f}  "
          f"(common_ground={s['common_ground']:.2f}, complementarity={s['complementarity']:.2f}, "
          f"useful_diversity={s['useful_diversity']:.2f}, session_continuity={s['session_continuity']:.2f}, "
          f"friction={s['friction']:.2f})")
print(f"\nRationale:\n  {pod['rationale']}")
print(f"\nConversation starter:\n  {pod['starter']}")
print(f"\nParticipantes: {pod['participant_ids']}")


## 8. Consultar o pod de um participante específico

In [ ]:
PARTICIPANT_ID = "122"

found = None
for pid, pod in pods_index.items():
    if PARTICIPANT_ID in pod["participant_ids"]:
        found = pid
        break

if found:
    print(f"Participante {PARTICIPANT_ID} esta no pod {found}\n")
    pod = pods_index[found]
    print(f"Rationale: {pod['rationale']}")
    print(f"Starter: {pod['starter']}")
else:
    print(f"Participante {PARTICIPANT_ID} nao esta em nenhum pod guiado "
          f"(pode ser opt-out / Open Networking).")


## 9. Relatório de explainability completo (markdown)

In [ ]:
report = build_explainability_report(pods_by_zone, meta)
with open(f"{OUT_DIR}/explainability_report.md", "w", encoding="utf-8") as f:
    f.write(report)

from IPython.display import Markdown, display
display(Markdown(report[:4000] + "\n\n...(ver arquivo completo em out/explainability_report.md)"))


## 10. Recalibrar pesos e re-rodar (opcional)

Os pesos de `ConversationValue` (seção 8.6 do dossiê) vivem em `scoring.MATCHING_WEIGHTS`.
Você pode ajustá-los aqui e re-rodar a partir da célula 4 sem editar os `.py`.

In [ ]:
import scoring

# exemplo: dar mais peso a session_continuity
# scoring.MATCHING_WEIGHTS["session_continuity"] = 1.2

print(scoring.MATCHING_WEIGHTS)
